# Stage 1: Cassava Disease Recognition - Qwen2.5-VL-7B

## Migration Gemma 3n → Qwen2.5-VL-7B

### Insights transférés de Gemma V3.1:
- ✅ LoRA conservatif: r=8, alpha=8, dropout=0
- ✅ Prompt multiple-choice (A/B/C/D/E)
- ✅ Data augmentation (albumentations)
- ✅ Oversampling CBSD 3x, Healthy 2x
- ✅ Fonction extract_prediction corrigée
- ✅ Weight decay=0.01, LR=2e-5

### Résultats Gemma V3.1 (baseline):
- CMD: 89.4%
- CGM: 81.4%
- Healthy: 40.3%
- CBB: 34.0%
- CBSD: 2.9% ❌ (à améliorer!)

### Avantages attendus de Qwen2.5-VL-7B:
- Meilleure capacité visuelle (7B vs 2B)
- Meilleur instruction-following
- Architecture plus récente (2024)

---

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


In [2]:
# ============================================================
# CELL 2: CONFIGURATION QWEN2.5-VL-7B
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "stage1"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage1_cassava_qwen25vl_7b"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 🔥 MODÈLE QWEN2.5-VL-7B
# ============================================================
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)  # Qwen supporte variable, mais on garde 512

# ============================================================
# LoRA Configuration (transférée de Gemma V3.1)
# ============================================================
LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 8,           # ratio 1:1 (conservatif)
    "lora_dropout": 0.0,       # 0 pour Unsloth
    "target_modules": ["q_proj", "v_proj"],  # Minimal
}

# ============================================================
# Training Configuration (transférée de Gemma V3.1)
# ============================================================
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,     # Peut réduire à 1 si OOM
    "gradient_accumulation_steps": 8,     # Effective batch = 16
    "num_train_epochs": 3,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 0.5,
    "logging_steps": 25,
    "save_steps": 200,
    "eval_steps": 100,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

# ============================================================
# Classes et Oversampling (transféré de Gemma V3.2)
# ============================================================
CLASS_NAMES = [
    "Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)",
    "Healthy"
]

# Oversampling pour rééquilibrer (CBSD était à 2.9%)
OVERSAMPLING_RATIOS = {
    "Cassava Bacterial Blight (CBB)": 1,
    "Cassava Brown Streak Disease (CBSD)": 3,  # 3x pour CBSD
    "Cassava Green Mottle (CGM)": 1,
    "Cassava Mosaic Disease (CMD)": 1,
    "Healthy": 2  # 2x pour Healthy
}

print(f"{'='*60}")
print(f"CONFIGURATION QWEN2.5-VL-7B")
print(f"{'='*60}")
print(f"\nModèle: {MODEL_NAME}")
print(f"\nLoRA: r={LORA_CONFIG['r']}, alpha={LORA_CONFIG['lora_alpha']}")
print(f"Modules: {LORA_CONFIG['target_modules']}")
print(f"\nTraining:")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  - Batch: {TRAINING_CONFIG['per_device_train_batch_size']} x {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"\nOversampling:")
for cls, ratio in OVERSAMPLING_RATIOS.items():
    if ratio > 1:
        print(f"  - {cls}: {ratio}x")
print(f"{'='*60}")

CONFIGURATION QWEN2.5-VL-7B

Modèle: unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit

LoRA: r=8, alpha=8
Modules: ['q_proj', 'v_proj']

Training:
  - LR: 2e-05
  - Epochs: 3
  - Batch: 2 x 8

Oversampling:
  - Cassava Brown Streak Disease (CBSD): 3x
  - Healthy: 2x


In [3]:
# ============================================================
# CELL 3: PROMPT ET EXTRACTION (identique à Gemma)
# ============================================================

CONSTRAINED_PROMPT_TEMPLATE = """Look at this cassava leaf image and identify the disease.

Choose EXACTLY ONE option:
A) Cassava Bacterial Blight (CBB)
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only."""

RESPONSE_TEMPLATES = {
    "Cassava Bacterial Blight (CBB)": "A) Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)": "B) Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)": "C) Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)": "D) Cassava Mosaic Disease (CMD)",
    "Healthy": "E) Healthy"
}

LETTER_TO_CLASS = {
    "A": "Cassava Bacterial Blight (CBB)",
    "B": "Cassava Brown Streak Disease (CBSD)",
    "C": "Cassava Green Mottle (CGM)",
    "D": "Cassava Mosaic Disease (CMD)",
    "E": "Healthy"
}

def extract_prediction(response: str) -> str:
    """Extrait la prédiction - VERSION CORRIGÉE (de Gemma V3.1)."""
    
    # Nettoyer: garder après "assistant" ou "model"
    for delimiter in ["assistant\n", "assistant", "model\n", "model"]:
        if delimiter in response.lower():
            response = response.lower().split(delimiter)[-1]
            break
    
    response = response.strip()
    
    # Chercher "X)" au début
    for letter in ["A", "B", "C", "D", "E"]:
        if response.upper().startswith(f"{letter})") or response.upper().startswith(f"{letter} )"):
            return LETTER_TO_CLASS[letter]
    
    # Fallback: regex pour trouver "X)" n'importe où
    match = re.search(r'\b([A-E])\)', response, re.IGNORECASE)
    if match:
        return LETTER_TO_CLASS[match.group(1).upper()]
    
    # Fallback: chercher les noms de maladies
    response_lower = response.lower()
    if "cbb" in response_lower or "bacterial blight" in response_lower:
        return "Cassava Bacterial Blight (CBB)"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "Cassava Brown Streak Disease (CBSD)"
    elif "cgm" in response_lower or "green mottle" in response_lower:
        return "Cassava Green Mottle (CGM)"
    elif "cmd" in response_lower or "mosaic" in response_lower:
        return "Cassava Mosaic Disease (CMD)"
    elif "healthy" in response_lower:
        return "Healthy"
    
    return "Unknown"

print("✅ Prompt et extraction configurés (identiques à Gemma)")

✅ Prompt et extraction configurés (identiques à Gemma)


In [4]:
# ============================================================
# CELL 4: CHARGEMENT MODÈLE QWEN2.5-VL-7B
# ============================================================

print("\n" + "="*60)
print("LOADING QWEN2.5-VL-7B")
print("="*60)

# Charger le modèle
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Ajouter LoRA
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
    use_rslora=False,
    use_gradient_checkpointing="unsloth"
)

# Charger le processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✅ Modèle chargé: {MODEL_NAME}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print("="*60)


LOADING QWEN2.5-VL-7B
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.90G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]


✅ Modèle chargé: unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit
Trainable: 2,523,136 / 5,032,045,568 (0.0501%)


In [5]:
# ============================================================
# CELL 5: CHARGEMENT DONNÉES
# ============================================================

print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

def load_json_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

cassava_train = load_json_dataset(DATA_DIR / "cassava_train_balanced.json")
cassava_val = load_json_dataset(DATA_DIR / "cassava_val.json")
cassava_test = load_json_dataset(DATA_DIR / "cassava_test.json")

print(f"Train: {len(cassava_train):,}")
print(f"Val: {len(cassava_val):,}")
print(f"Test: {len(cassava_test):,}")


LOADING DATA
Train: 29,276
Val: 3,612
Test: 3,619


In [6]:
# ============================================================
# CELL 6: FORMATAGE ET OVERSAMPLING
# ============================================================

def extract_image_path(text):
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def get_class_from_response(response: str) -> str:
    for class_name in CLASS_NAMES:
        if class_name in response:
            return class_name
    return "Unknown"

def get_class_label(sample) -> str:
    for turn in sample['conversations']:
        if turn['from'] == 'assistant':
            return get_class_from_response(turn['value'])
    return "Unknown"

def format_sample_for_qwen(sample):
    """Formate un sample pour Qwen2.5-VL."""
    conversations = sample['conversations']
    
    # Extraire le chemin de l'image
    image_path = None
    for turn in conversations:
        if '<img>' in turn['value']:
            image_path = extract_image_path(turn['value'])
            break
    
    # Obtenir la classe
    class_label = get_class_label(sample)
    
    # Format Qwen - utilise le même format que Gemma
    formatted_conversations = [
        {
            "role": "user",
            "content": f"Picture 1: <image>\n{CONSTRAINED_PROMPT_TEMPLATE}"
        },
        {
            "role": "assistant",
            "content": RESPONSE_TEMPLATES.get(class_label, "E) Healthy")
        }
    ]
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id'],
        "class_label": class_label,
        "class_idx": CLASS_NAMES.index(class_label) if class_label in CLASS_NAMES else 4
    }

# Formater
print("\nFormatting data...")
train_formatted_original = [format_sample_for_qwen(s) for s in tqdm(cassava_train, desc="Train")]
val_formatted = [format_sample_for_qwen(s) for s in tqdm(cassava_val, desc="Val")]

# Distribution originale
print("\nDistribution originale:")
original_dist = Counter(s['class_label'] for s in train_formatted_original)
for cls, count in original_dist.most_common():
    print(f"  {cls}: {count}")

# Appliquer oversampling
print("\nApplying oversampling...")
samples_by_class = defaultdict(list)
for sample in train_formatted_original:
    samples_by_class[sample['class_label']].append(sample)

train_formatted = []
for class_name, samples in samples_by_class.items():
    ratio = OVERSAMPLING_RATIOS.get(class_name, 1)
    oversampled = samples * ratio
    train_formatted.extend(oversampled)
    if ratio > 1:
        print(f"  {class_name}: {len(samples)} → {len(oversampled)} ({ratio}x)")

random.seed(42)
random.shuffle(train_formatted)

print(f"\n✅ Total après oversampling: {len(train_formatted):,}")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)


Formatting data...


Val: 100%|██████████| 3612/3612 [00:00<00:00, 695616.24it/s]


Distribution originale:
  Cassava Mosaic Disease (CMD): 12000
  Cassava Brown Streak Disease (CBSD): 5296
  Cassava Green Mottle (CGM): 5054
  Cassava Bacterial Blight (CBB): 3717
  Healthy: 3209

Applying oversampling...
  Cassava Brown Streak Disease (CBSD): 5296 → 15888 (3x)
  Healthy: 3209 → 6418 (2x)

✅ Total après oversampling: 43,077


In [7]:
# ============================================================
# CELL 7: DATA COLLATOR POUR QWEN
# ============================================================

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
    train_transforms = A.Compose([
        A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, 
                        min_holes=1, min_height=8, min_width=8, p=0.2),
    ])
    print("✅ Albumentations - augmentation activée")
except ImportError:
    HAS_ALBUMENTATIONS = False
    train_transforms = None
    print("⚠️ Albumentations non disponible")

@dataclass
class QwenVLDataCollator:
    """Data collator pour Qwen2.5-VL."""
    processor: Any
    transforms: Any = None
    is_training: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []
        batch_class_idx = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert('RGB')
                    
                    # Augmentation
                    if self.is_training and self.transforms is not None:
                        image_np = np.array(image)
                        augmented = self.transforms(image=image_np)
                        image = Image.fromarray(augmented['image'])
                    
                    # Resize
                    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                except Exception as e:
                    print(f"Error: {e}")
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            else:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image] if image else None)
            batch_class_idx.append(feature.get('class_idx', 4))

            # Construire messages pour Qwen
            conversations = feature['conversations']
            messages = []
            
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and image is not None and len(messages) == 0:
                    # Premier message user avec image
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": content.replace('<image>', '').replace('Picture 1:', '').strip()}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        # Appliquer le chat template
        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            # Texte complet (user + assistant)
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            # Texte user seulement (pour masquer dans labels)
            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        # Process avec le processor Qwen
        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )

        # Masquer les labels (user tokens = -100)
        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=1024
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-5] = -100

        batch["labels"] = labels
        batch["class_idx"] = torch.tensor(batch_class_idx)

        return batch

train_collator = QwenVLDataCollator(processor=processor, transforms=train_transforms, is_training=True)
eval_collator = QwenVLDataCollator(processor=processor, transforms=None, is_training=False)

print("✅ Data collators créés")

✅ Albumentations - augmentation activée
✅ Data collators créés


/tmp/ipykernel_1510245/2324420326.py:17: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
/tmp/ipykernel_1510245/2324420326.py:18: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,


In [8]:
# ============================================================
# CELL 8: VÉRIFICATION DATA COLLATOR
# ============================================================

print("\n" + "="*60)
print("TESTING DATA COLLATOR")
print("="*60)

# Test avec quelques samples
test_features = [train_dataset[i] for i in range(2)]

try:
    test_batch = train_collator(test_features)
    print(f"✅ Batch créé avec succès!")
    print(f"   - input_ids shape: {test_batch['input_ids'].shape}")
    print(f"   - labels shape: {test_batch['labels'].shape}")
    
    # Vérifier tokens actifs
    for i in range(len(test_features)):
        active = (test_batch['labels'][i] != -100).sum().item()
        total = (test_batch['labels'][i] != -100).shape[0]
        print(f"   - Sample {i}: {active} tokens actifs")
except Exception as e:
    print(f"❌ Erreur: {e}")
    import traceback
    traceback.print_exc()


TESTING DATA COLLATOR
✅ Batch créé avec succès!
   - input_ids shape: torch.Size([2, 437])
   - labels shape: torch.Size([2, 437])
   - Sample 0: 14 tokens actifs
   - Sample 1: 14 tokens actifs


In [9]:
# ============================================================
# CELL 9: TRAINER
# ============================================================

class CassavaTrainer(Trainer):
    """Trainer simplifié."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop('class_idx', None)
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("✅ CassavaTrainer configuré")

✅ CassavaTrainer configuré


In [10]:
# ============================================================
# CELL 10: CONFIGURATION TRAINING
# ============================================================

print("\n" + "="*60)
print("CONFIGURING TRAINER")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # Training
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    
    # Regularization
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    # Logging
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=3,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    # Precision
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    
    # Optimizer
    optim="adamw_8bit",
    
    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Misc
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# Early stopping (moins agressif)
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=10,
    early_stopping_threshold=0.001
)

trainer = CassavaTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=train_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch * TRAINING_CONFIG['num_train_epochs']

print(f"\nConfiguration:")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print("="*60)


CONFIGURING TRAINER

Configuration:
  - Train samples: 43,077
  - Effective batch: 16
  - Total steps: ~8,076
  - LR: 2e-05
  - Epochs: 3


In [11]:
# ============================================================
# CELL 11: ENTRAÎNEMENT
# ============================================================

print("\n" + "="*60)
print("STARTING TRAINING - QWEN2.5-VL-7B")
print("="*60)
print(f"\n🎯 Objectifs (basés sur Gemma V3.1):")
print(f"  - CMD: >85% (Gemma: 89%)")
print(f"  - CGM: >70% (Gemma: 81%)")
print(f"  - CBB: >30% (Gemma: 34%)")
print(f"  - Healthy: >40% (Gemma: 40%)")
print(f"  - CBSD: >30% (Gemma: 2.9% - à améliorer!)")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 43,077 | Num Epochs = 3 | Total steps = 8,079
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 2,523,136 of 8,294,689,792 (0.03% trained)



STARTING TRAINING - QWEN2.5-VL-7B

🎯 Objectifs (basés sur Gemma V3.1):
  - CMD: >85% (Gemma: 89%)
  - CGM: >70% (Gemma: 81%)
  - CBB: >30% (Gemma: 34%)
  - Healthy: >40% (Gemma: 40%)
  - CBSD: >30% (Gemma: 2.9% - à améliorer!)

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,9.634300,0.153705
200,8.761200,0.149874
300,5.452000,0.137361
400,2.606900,0.127535
500,0.879400,0.118130
600,0.762400,0.103739
700,0.723400,0.106678
800,0.661500,0.102053
900,0.610300,0.099483
1000,0.638500,0.089027


Error: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_00d853a6-e753-4e53-8e5f-77a36504d6d6.jpg'
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_003/uuid_6d757e37-bafa-40bc-8769-f1f4c12e3d8f.jpg'
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_004/uuid_cf76a517-1492-4874-bd34-a2a389f78426.jpg'
Error: broken data stream when reading image file
Error: broken data stream when reading image file
Error: cannot identify image file '/home/hounfodjid

In [12]:
# ============================================================
# CELL 12: SAUVEGARDE
# ============================================================

print("\nSaving model...")
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Sauvegarder config
config = {
    "model_name": MODEL_NAME,
    "lora_config": LORA_CONFIG,
    "training_config": TRAINING_CONFIG,
    "oversampling_ratios": OVERSAMPLING_RATIOS,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
}

with open(CHECKPOINT_DIR / "training_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Saved to: {final_model_path}")


Saving model...
✅ Saved to: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model


In [13]:
# ============================================================
# CELL 13: ÉVALUATION
# ============================================================

print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

# Formatter test set
test_formatted = [format_sample_for_qwen(s) for s in tqdm(cassava_test, desc="Formatting test")]

# Mode inference
FastVisionModel.for_inference(model)

def predict_single(sample, max_new_tokens=50):
    """Prédit pour un sample."""
    image_path = sample['image_path']
    
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Error", ""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": CONSTRAINED_PROMPT_TEMPLATE}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted_class = extract_prediction(response)
    
    return predicted_class, response

# Quick test
print("\n--- Quick Test (100 samples) ---")
quick_samples = random.sample(test_formatted, 100)
quick_results = []

for sample in tqdm(quick_samples, desc="Quick test"):
    pred, raw = predict_single(sample)
    quick_results.append({
        'gt': sample['class_label'],
        'pred': pred,
        'correct': pred == sample['class_label'],
        'raw': raw
    })

correct = sum(1 for r in quick_results if r['correct'])
print(f"\n📊 Quick Accuracy: {correct}%")

print("\n📈 Par classe:")
for cls in CLASS_NAMES:
    cls_samples = [r for r in quick_results if r['gt'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        status = "✅" if acc >= 30 else "❌"
        print(f"  {status} {cls}: {acc:.1f}% ({cls_correct}/{len(cls_samples)})")

# Vérifier hallucinations
unknown = sum(1 for r in quick_results if r['pred'] == 'Unknown')
print(f"\n🔍 Hallucinations: {unknown}")


EVALUATION ON TEST SET


Formatting test: 100%|██████████| 3619/3619 [00:00<00:00, 680589.44it/s]



--- Quick Test (100 samples) ---


Quick test: 100%|██████████| 100/100 [00:49<00:00,  2.01it/s]


📊 Quick Accuracy: 85%

📈 Par classe:
  ✅ Cassava Bacterial Blight (CBB): 100.0% (5/5)
  ✅ Cassava Brown Streak Disease (CBSD): 81.0% (17/21)
  ✅ Cassava Green Mottle (CGM): 62.5% (5/8)
  ✅ Cassava Mosaic Disease (CMD): 91.1% (51/56)
  ✅ Healthy: 70.0% (7/10)

🔍 Hallucinations: 0


In [14]:
# ============================================================
# CELL 14: DEBUG - VOIR LES RÉPONSES BRUTES
# ============================================================

print("\n=== RÉPONSES BRUTES DU MODÈLE ===")

debug_samples = random.sample(quick_results, 10)
for i, r in enumerate(debug_samples):
    print(f"\n[{i+1}] GT: {r['gt']}")
    print(f"    Pred: {r['pred']}")
    # Afficher les 200 derniers caractères de la réponse
    raw = r['raw']
    print(f"    Raw (fin): '...{raw[-200:]}'")


=== RÉPONSES BRUTES DU MODÈLE ===

[1] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease (CMD)
    Raw (fin): '...
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only.
assistant
D) Cassava Mosaic Disease (CMD)'

[2] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease (CMD)
    Raw (fin): '...
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only.
assistant
D) Cassava Mosaic Disease (CMD)'

[3] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease (CMD)
    Raw (fin): '...
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only.
assistant
D) Cassava Mosaic Disease (CMD)'

[4] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease 

In [15]:
# ============================================================
# CELL 15: ÉVALUATION COMPLÈTE (optionnel)
# ============================================================

RUN_FULL_EVAL = True

if RUN_FULL_EVAL:
    print("\n" + "="*60)
    print("FULL EVALUATION")
    print("="*60)
    print(f"⏱️ Temps estimé: ~{len(test_formatted) / 60:.0f} minutes")
    
    all_results = []
    
    for sample in tqdm(test_formatted, desc="Full test"):
        pred, _ = predict_single(sample)
        all_results.append({
            'gt': sample['class_label'],
            'pred': pred,
            'correct': pred == sample['class_label']
        })
    
    # Métriques
    total = len(all_results)
    correct = sum(1 for r in all_results if r['correct'])
    
    print(f"\n" + "="*60)
    print(f"RÉSULTATS FINAUX - QWEN2.5-VL-7B")
    print(f"="*60)
    print(f"\n📊 ACCURACY GLOBALE: {correct/total*100:.2f}% ({correct}/{total})")
    
    print("\n📈 PAR CLASSE (vs Gemma V3.1):")
    gemma_results = {
        "Cassava Bacterial Blight (CBB)": 34.0,
        "Cassava Brown Streak Disease (CBSD)": 2.9,
        "Cassava Green Mottle (CGM)": 81.4,
        "Cassava Mosaic Disease (CMD)": 89.4,
        "Healthy": 40.3
    }
    
    class_metrics = {}
    for cls in CLASS_NAMES:
        cls_samples = [r for r in all_results if r['gt'] == cls]
        if cls_samples:
            cls_correct = sum(1 for r in cls_samples if r['correct'])
            acc = cls_correct / len(cls_samples) * 100
            class_metrics[cls] = acc
            gemma_acc = gemma_results.get(cls, 0)
            diff = acc - gemma_acc
            diff_str = f"+{diff:.1f}" if diff > 0 else f"{diff:.1f}"
            status = "✅" if acc >= 30 else "❌"
            print(f"  {status} {cls}: {acc:.1f}% ({diff_str}% vs Gemma)")
    
    # Hallucinations
    unknown = sum(1 for r in all_results if r['pred'] == 'Unknown')
    print(f"\n🔍 Hallucinations: {unknown} ({unknown/total*100:.2f}%)")
    
    # Sauvegarder
    results = {
        'model': 'Qwen2.5-VL-7B',
        'global_accuracy': correct/total*100,
        'class_metrics': class_metrics,
        'comparison_vs_gemma': {cls: class_metrics.get(cls, 0) - gemma_results.get(cls, 0) for cls in CLASS_NAMES},
        'hallucinations': unknown
    }
    
    with open(CHECKPOINT_DIR / "test_results.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✅ Résultats sauvegardés")
    
    # Verdict
    print("\n" + "="*60)
    print("VERDICT: QWEN vs GEMMA")
    print("="*60)
    
    qwen_better = sum(1 for cls in CLASS_NAMES if class_metrics.get(cls, 0) > gemma_results.get(cls, 0))
    if qwen_better >= 3:
        print(f"🏆 QWEN GAGNE! Meilleur sur {qwen_better}/5 classes")
    elif qwen_better >= 2:
        print(f"🤝 ÉGALITÉ - Qwen meilleur sur {qwen_better}/5 classes")
    else:
        print(f"😕 GEMMA reste meilleur sur {5-qwen_better}/5 classes")


FULL EVALUATION
⏱️ Temps estimé: ~60 minutes


Full test: 100%|██████████| 3619/3619 [26:06<00:00,  2.31it/s]


RÉSULTATS FINAUX - QWEN2.5-VL-7B

📊 ACCURACY GLOBALE: 80.52% (2914/3619)

📈 PAR CLASSE (vs Gemma V3.1):
  ✅ Cassava Bacterial Blight (CBB): 55.1% (+21.1% vs Gemma)
  ✅ Cassava Brown Streak Disease (CBSD): 84.0% (+81.1% vs Gemma)
  ✅ Cassava Green Mottle (CGM): 56.5% (-24.9% vs Gemma)
  ✅ Cassava Mosaic Disease (CMD): 86.0% (-3.4% vs Gemma)
  ✅ Healthy: 75.4% (+35.1% vs Gemma)

🔍 Hallucinations: 0 (0.00%)

✅ Résultats sauvegardés

VERDICT: QWEN vs GEMMA
🏆 QWEN GAGNE! Meilleur sur 3/5 classes


In [ ]:
# ============================================================
# CELL 16: COMPARAISON FINALE
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║           COMPARAISON GEMMA 3n vs QWEN2.5-VL-7B              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Classe          │  Gemma 3n  │  Qwen 7B  │  Différence     ║
║  ─────────────────────────────────────────────────────────── ║
║  CMD             │   89.4%    │    ??%    │     ??          ║
║  CGM             │   81.4%    │    ??%    │     ??          ║
║  Healthy         │   40.3%    │    ??%    │     ??          ║
║  CBB             │   34.0%    │    ??%    │     ??          ║
║  CBSD            │    2.9%    │    ??%    │     ??          ║
║  ─────────────────────────────────────────────────────────── ║
║  Global          │   65.0%    │    ??%    │     ??          ║
║                                                              ║
║  (Les ?? seront remplis après l'évaluation)                  ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")